In [5]:
from getpass import getpass
from collections import defaultdict

import pandas as pd

from mp_api.client import MPRester
from pymatgen.core import Composition
from pymatgen.analysis.structure_matcher import StructureMatcher
from getpass import getpass

MP_API_KEY = "qKRsC5yQCETJ7HimEcjR9AcCC9LD3pN6"

if not MP_API_KEY:
    raise ValueError("No API key was entered.")



In [11]:
test_formula = "SiO2"

fields = [
    "material_id",
    "formula_pretty",
    "structure",
    "symmetry",
    "energy_above_hull",
    "formation_energy_per_atom",
    "database_IDs",
    "volume",
    "nsites",
]

with MPRester(MP_API_KEY) as mpr:
    docs = mpr.materials.summary.search(
        formula=test_formula,
        fields=fields,
    )

print(f"Number of Materials Project entries returned: {len(docs)}")

Retrieving SummaryDoc documents: 100%|██████████| 322/322 [00:00<00:00, 6311055.55it/s]

Number of Materials Project entries returned: 322


In [12]:
assert len(docs) > 0, f"No Materials Project entries found for {test_formula}"

results = []

for doc in docs:
    structure = doc.structure
    symmetry = doc.symmetry

    results.append({
        "material_id": str(doc.material_id),
        "formula": doc.formula_pretty,
        "reduced_formula_from_structure": structure.composition.reduced_formula,
        "space_group": symmetry.symbol if symmetry else None,
        "space_group_number": symmetry.number if symmetry else None,
        "crystal_system": symmetry.crystal_system if symmetry else None,
        "energy_above_hull": doc.energy_above_hull,
        "formation_energy_per_atom": doc.formation_energy_per_atom,
        "volume": doc.volume,
        "nsites": doc.nsites,
        "structure": structure,
        "database_ids": doc.database_IDs,
    })

results_df = pd.DataFrame(results)

display(
    results_df.drop(columns=["structure", "database_ids"])
    .sort_values(["space_group", "material_id"])
)

,material_id,formula,reduced_formula_from_structure,space_group,space_group_number,crystal_system,energy_above_hull,formation_energy_per_atom,volume,nsites
90,mp-554573,SiO2,SiO2,Ama2,40,Orthorhombic,0.007466,-3.265365,180.890683,12
213,mp-561181,SiO2,SiO2,Amm2,38,Orthorhombic,0.021043,-3.251788,1851.069445,84
253,mp-600055,SiO2,SiO2,Amm2,38,Orthorhombic,0.019013,-3.253819,1605.950849,84
129,mp-556553,SiO2,SiO2,C2,5,Monoclinic,0.085702,-3.187130,155.921688,12
134,mp-556662,SiO2,SiO2,C2,5,Monoclinic,0.067856,-3.204975,171.581204,12
...,...,...,...,...,...,...,...,...,...,...
279,mp-638035,SiO2,SiO2,R-3c,167,Trigonal,1.830344,-1.442487,185.791202,18
66,mp-17909,SiO2,SiO2,R-3m,166,Trigonal,0.021387,-3.251444,791.792407,36
188,mp-559605,SiO2,SiO2,R-3m,166,Trigonal,0.238962,-3.033870,307.137553,18
217,mp-561488,SiO2,SiO2,R-3m,166,Trigonal,0.512761,-2.760070,685.690817,36


In [13]:
returned_formulas = set(results_df["reduced_formula_from_structure"])

print("Requested formula:", Composition(test_formula).reduced_formula)
print("Returned reduced formulas:", returned_formulas)

assert returned_formulas == {Composition(test_formula).reduced_formula}

Requested formula: SiO2
Returned reduced formulas: {'SiO2'}


In [15]:
import time
import pandas as pd

# Compare a small subset during the initial test
max_test_structures = 25

if len(results_df) > max_test_structures:
    test_results_df = (
        results_df
        .sample(n=max_test_structures, random_state=42)
        .reset_index(drop=True)
    )
else:
    test_results_df = results_df.reset_index(drop=True)

n = len(test_results_df)
total_comparisons = n * (n - 1) // 2

print(f"Structures selected: {n}")
print(f"Pairwise comparisons: {total_comparisons}")

comparison_rows = []
start_time = time.perf_counter()

for count, (i, j) in enumerate(
    ((i, j) for i in range(n) for j in range(i + 1, n)),
    start=1
):
    structure_i = test_results_df.loc[i, "structure"]
    structure_j = test_results_df.loc[j, "structure"]

    structures_match = matcher.fit(structure_i, structure_j)

    comparison_rows.append({
        "material_id_1": test_results_df.loc[i, "material_id"],
        "material_id_2": test_results_df.loc[j, "material_id"],
        "formula_1": test_results_df.loc[i, "formula"],
        "formula_2": test_results_df.loc[j, "formula"],
        "space_group_1": test_results_df.loc[i, "space_group"],
        "space_group_2": test_results_df.loc[j, "space_group"],
        "structures_match": structures_match,
    })

    if count == 1 or count % 50 == 0 or count == total_comparisons:
        elapsed = time.perf_counter() - start_time
        print(
            f"Completed {count}/{total_comparisons} "
            f"comparisons ({elapsed:.1f} seconds)"
        )

comparison_df = pd.DataFrame(comparison_rows)

print("Comparison complete.")
display(comparison_df)

Structures selected: 25
Pairwise comparisons: 300
Completed 1/300 comparisons (0.0 seconds)
Completed 50/300 comparisons (1.8 seconds)
Completed 100/300 comparisons (3.5 seconds)
Completed 150/300 comparisons (6.6 seconds)
Completed 200/300 comparisons (9.8 seconds)
Completed 250/300 comparisons (11.2 seconds)
Completed 300/300 comparisons (13.6 seconds)
Comparison complete.


,material_id_1,material_id_2,formula_1,formula_2,space_group_1,space_group_2,structures_match
0,mp-558374,mp-556591,SiO2,SiO2,Pa-3,Fmmm,False
1,mp-558374,mp-560152,SiO2,SiO2,Pa-3,P-42_1c,False
2,mp-558374,mp-1179508,SiO2,SiO2,Pa-3,P3_221,False
3,mp-558374,mp-555497,SiO2,SiO2,Pa-3,Ia-3,False
4,mp-558374,mp-556132,SiO2,SiO2,Pa-3,Pcca,False
...,...,...,...,...,...,...,...
295,mp-8602,mp-667368,SiO2,SiO2,P6/mmm,Immm,False
296,mp-8602,mp-644923,SiO2,SiO2,P6/mmm,I-4m2,False
297,mp-6945,mp-667368,SiO2,SiO2,P4_12_12,Immm,False
298,mp-6945,mp-644923,SiO2,SiO2,P4_12_12,I-4m2,False


In [17]:
# Group equivalent structures using the comparison results from Cell 5

n = len(test_results_df)
parent = list(range(n))


def find(index):
    """Find the representative of a structure group."""
    while parent[index] != index:
        parent[index] = parent[parent[index]]
        index = parent[index]
    return index


def union(index_a, index_b):
    """Join two structures into the same group."""
    root_a = find(index_a)
    root_b = find(index_b)

    if root_a != root_b:
        parent[root_b] = root_a


# Map Materials Project IDs to row positions
id_to_index = {
    material_id: index
    for index, material_id in enumerate(test_results_df["material_id"])
}


# Join structures that Cell 5 identified as equivalent
for row in comparison_df.itertuples(index=False):
    if row.structures_match:
        index_a = id_to_index[row.material_id_1]
        index_b = id_to_index[row.material_id_2]
        union(index_a, index_b)


# Assign readable group numbers
root_to_group = {}
next_group_number = 1
group_numbers = []

for index in range(n):
    root = find(index)

    if root not in root_to_group:
        root_to_group[root] = next_group_number
        next_group_number += 1

    group_numbers.append(root_to_group[root])


# Add group assignments to the test dataframe
test_results_df = test_results_df.copy()
test_results_df["structure_group"] = group_numbers

print(f"Equivalent structure groups found: {test_results_df['structure_group'].nunique()}")

display(
    test_results_df
    .drop(columns=["structure", "database_ids"])
    .sort_values(["structure_group", "material_id"])
)

Equivalent structure groups found: 25


,material_id,formula,reduced_formula_from_structure,space_group,space_group_number,crystal_system,energy_above_hull,formation_energy_per_atom,volume,nsites,structure_group
0,mp-558374,SiO2,SiO2,Pa-3,205,Cubic,0.039755,-3.233076,1136.598478,72,1
1,mp-556591,SiO2,SiO2,Fmmm,69,Orthorhombic,0.020681,-3.252150,1073.182049,54,2
2,mp-560152,SiO2,SiO2,P-42_1c,114,Tetragonal,0.103626,-3.169206,697.877982,48,3
3,mp-1179508,SiO2,SiO2,P3_221,154,Trigonal,0.336020,-2.936811,95.656355,9,4
4,mp-555497,SiO2,SiO2,Ia-3,206,Cubic,0.255605,-3.017227,883.495836,72,5
5,mp-556132,SiO2,SiO2,Pcca,54,Orthorhombic,0.196783,-3.076048,273.777414,24,6
6,mp-600060,SiO2,SiO2,Pnma,62,Orthorhombic,0.017455,-3.255376,2768.784291,144,7
7,mp-557814,SiO2,SiO2,Pm-3n,223,Cubic,0.711423,-2.561409,2295.777288,144,8
8,mp-600006,SiO2,SiO2,P4/mcc,124,Tetragonal,0.034152,-3.238679,1009.527232,60,9
9,mp-667448,SiO2,SiO2,I-4,82,Tetragonal,0.243598,-3.029233,2738.861958,144,10
